# 21 — ALICE-style TCR-neighborhood enrichment (CD4 malignant subclones · cross-patient · reactive CD8)

ALICE (Pogorelyy et al., PNAS 2019): clonotypes are vertices, edges join CDR3s differing by
**≤1 amino acid**; flag clonotypes with **more neighbors than a generative VDJ-recombination
null predicts** (OLGA `humanTRB` Pgen × thymic-selection factor *Q*; Poisson test, BH-FDR).
Control-free single-snapshot variant — **no healthy background, no TCRNET, TRB only**, Pgen
marginalized over V/J (the atlas clone table has no V gene).

Three uses (see `CLONE_CNV_REVIEW.md` §4/§6):
1. **Subclone-family recovery** (CD4) — the dominant clone's ≤1-aa neighborhood = the malignant
   β-variant family the exact-match caller misses (Srinivas single-aa variants; branched evolution).
   *Caveat:* enrichment here is **somatic divergence, not antigen** — a sequence-family detector;
   disambiguate vs antigen with CNV context.
2. **Cross-patient convergence** (CD4 founders pooled) — shared antigen/superantigen? Expected
   clean negative (one clone/donor is sparse — underpowered).
3. **Reactive clusters** in the **CD8 infiltrate** of the same donors — antigen-driven TIL responses.

Load chain (cells 1–4) mirrors **nb18** and is CPU-light. The OLGA Pgen sweep (Use 1 / Use 3) is the
only heavy step — cached to parquet; run on the kernel or via `jobs/run_alice_pgen.sh`.

In [ ]:
# ============================================================
# Parameters — nb18 load chain + cohort filter + ALICE knobs.
# ============================================================
import hashlib, json
from pathlib import Path


def _resolve_nb_dir() -> Path:
    start = Path.cwd()
    for base in [start, *start.parents]:
        for sub in [Path("."), Path("notebooks/MF"), Path("scvi-tools-neural-nmf/notebooks/MF")]:
            cand = base / sub
            if cand.name == "MF" and (cand / "data").exists():
                return cand.resolve()
    raise FileNotFoundError(f"could not locate MF/data from {start}")


NB_DIR = _resolve_nb_dir(); print("NB_DIR =", NB_DIR)

# ---- inputs (same as nb18) ----
OBJ     = NB_DIR / "data" / "atlas_joint" / "skin_T_annotated_v2.h5ad"   # v2 CD4/CD8 re-annotation (cell_type_T2)
TCR_OBJ = NB_DIR / "data" / "atlas_joint" / "skin_T_tcr_annotated.h5ad"
LI_TCR  = NB_DIR / "data" / "Li2024_atlas" / "li2024_tcr_malignancy.parquet"

# ---- dominant-clone rule + cohort filter (identical to nb18) ----
FRAC_THRESH, RATIO_THRESH, EXPANDED_MIN = 0.05, 1.33, 2
# v2 annotation lives in obs["cell_type_T2"] = {CD4, CD4_Treg, CD8, NK}; CD4 part on CD4, CD8 part on CD8.
CD4_T_TYPES = ["CD4"]
CD8_T_TYPES = ["CD8"]
DROP_ENTITIES = {"MF_gamma_delta", "CD8_aggressive_epidermotropic_CTCL"}

# ---- ALICE params ----
ALPHA = 0.05
Q = None              # None -> calibrate per repertoire from the null bulk
SEED = 0

# ---- outputs (v2 = re-annotated cohort; kept separate from the cell_type_T run) ----
OUT_DIR = NB_DIR / "data" / "atlas_joint"
FIG_DIR = NB_DIR / "figures"; FIG_DIR.mkdir(exist_ok=True)
ALICE_CD4_PARQUET = OUT_DIR / "alice_cd4_per_donor_v2.parquet"
ALICE_CD8_PARQUET = OUT_DIR / "alice_cd8_v2.parquet"
USE1_CSV = OUT_DIR / "alice_use1_subclone_families_v2.csv"
USE2_CSV = OUT_DIR / "alice_use2_crosspatient_v2.csv"
USE3_CSV = OUT_DIR / "alice_use3_reactive_cd8_v2.csv"

In [ ]:
import sys, gc, importlib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(NB_DIR))
import atlas_join_helpers as H
import skin_T_cnv_helpers as C
import alice_helpers as A
for _m in (H, C, A):
    importlib.reload(_m)
np.random.seed(SEED)

In [ ]:
# Load cached TCR object, recompute dominant clone, apply nb18 sample selection.
adata = C.build_or_load_tcr_object(OBJ, TCR_OBJ, LI_TCR, H)   # ~12 GB cached

# v2 re-annotation: the cached TCR object predates cell_type_T2 -> merge it in by barcode.
import scanpy as sc
_v2 = sc.read_h5ad(OBJ, backed="r")
assert _v2.obs_names.equals(adata.obs_names), "v2 obs order mismatch"
adata.obs["cell_type_T2"] = _v2.obs["cell_type_T2"].astype(str).values
_v2.file.close(); del _v2
print("cell_type_T2:\n", adata.obs["cell_type_T2"].value_counts())

is_li = adata.obs["study"].astype(str).eq("li2024").to_numpy()
dom_tbl = C.recompute_dominant_clone(adata, H, is_li, FRAC_THRESH, RATIO_THRESH, EXPANDED_MIN)
clone_summary = C.clone_summary_table(adata, FRAC_THRESH, RATIO_THRESH)

META_COLS = ["study", "dataset", "disease", "disease_stage", "tissue", "entity", "sex", "tech"]
meta_cols = [c for c in META_COLS if c in adata.obs.columns]
donor_meta = (adata.obs[["donor", *meta_cols]].astype(str)
              .groupby("donor", observed=True)
              .agg(lambda s: ", ".join(sorted(s[s != "nan"].unique()))))
clone_summary = clone_summary.merge(donor_meta, on="donor", how="left")

# nb18 sample-selection: de-dup, drop non-ab-CD4 entities, size gate (>=300 except herrera).
drop = {}
if {"D5__MFIVB", "D1__P303"} <= set(clone_summary["donor"]):
    drop["D1__P303"] = "duplicate of D5__MFIVB"
for d in clone_summary.loc[clone_summary["entity"].isin(DROP_ENTITIES), "donor"]:
    drop.setdefault(d, "non-ab-CD4 entity")
small = (clone_summary["n_tcr_cells"] < 300) & (clone_summary["study"] != "herrera2021")
for d in clone_summary.loc[small, "donor"]:
    drop.setdefault(d, "n_tcr_cells < 300")
keep_donors = clone_summary.loc[~clone_summary["donor"].isin(drop), "donor"].tolist()
clone_summary = clone_summary[~clone_summary["donor"].isin(drop)].reset_index(drop=True)
adata = adata[adata.obs["donor"].isin(keep_donors)].copy()

# HC samples are benign by definition.
hc = adata.obs["disease"].astype(str).eq("HC").to_numpy()
adata.obs.loc[hc, ["tcr_is_malignant", "tcr_is_dominant_clone"]] = False
clone_summary.loc[clone_summary["disease"].eq("HC"), "malignant"] = False
print("kept donors:", len(keep_donors), "| cells:", adata.n_obs)

In [ ]:
# Per-clonotype tables: CD4 MF cohort (Use 1/2) and CD8 infiltrate of same donors (Use 3).
obs = adata.obs
cd4 = obs[obs["cell_type_T2"].isin(CD4_T_TYPES)]
cd8 = obs[obs["cell_type_T2"].isin(CD8_T_TYPES)]
clono_cd4 = A.clonotype_table(cd4, group="donor")
clono_cd8 = A.clonotype_table(cd8, group="donor")
print("CD4 clonotypes:", len(clono_cd4), "| donors:", clono_cd4["donor"].nunique(),
      "| founders:", int(clono_cd4["is_founder"].sum()))
print("CD8 clonotypes:", len(clono_cd8), "| donors:", clono_cd8["donor"].nunique())
clono_cd4.head()

## OLGA generative null + sanity check

`A.load_olga_trb()` loads the default human-TRB IGoR model shipped with OLGA. The expected number
of ≤1-aa neighbors of a clonotype is `N_unique × Q × Σ Pgen(single-mismatch variants)`; observed vs
expected is a Poisson survival test, BH-corrected per repertoire. *Q* is calibrated from the
low-degree null bulk (`A.calibrate_Q`); set `Q=...` above to fix it and sweep for sensitivity.

**Requires OLGA** — run once in this kernel: `%pip install olga`

In [ ]:
# %pip install olga    # uncomment + run once if olga is not importable
pgen_model = A.load_olga_trb()
pgen = A.make_pgen(pgen_model)

# sanity: the Rindler IVB single-aa-variant pair (a textbook ALICE edge / Srinivas subclone)
a, b = "CASSQDRALENTIYF", "CASSQDRTLENTIYF"
print("pgen(a) =", pgen(a), " pgen(b) =", pgen(b))
print("b is a single-mismatch variant of a:", b in set(A.one_mismatch_variants(a)))
print("graph edge a-b:", A.neighbor_graph([a, b]).has_edge(a, b))

## Use 1 — per-patient malignant subclone-family recovery

For each CD4 donor: run ALICE on the full clonotype table, then take the ≤1-aa connected component
containing the dominant founder = the **malignant subclone family**. Compare the tumor fraction it
captures against the exact-match founder fraction. **Heavy** (OLGA Pgen sweep) — cached.

In [ ]:
# Heavy: OLGA Pgen sweep per CD4 donor (only clonotypes with >=1 observed neighbor). Cached.
# Stale-cache guard: recompute when the cached donor set != the current clono_cd4 donor set.
alice_cd4 = pd.read_parquet(ALICE_CD4_PARQUET) if ALICE_CD4_PARQUET.exists() else None
if alice_cd4 is None or set(alice_cd4["donor"]) != set(clono_cd4["donor"]):
    if alice_cd4 is not None:
        print("CD4 cache stale (donor set changed) -> recompute")
    alice_cd4 = A.run_alice_by_group(clono_cd4, pgen, group="donor", Q=Q, alpha=ALPHA)
    alice_cd4.to_parquet(ALICE_CD4_PARQUET, index=False)
    print("wrote", ALICE_CD4_PARQUET)
else:
    print("loaded cached", ALICE_CD4_PARQUET)
print("CD4 donors:", alice_cd4["donor"].nunique(),
      "| significant clonotypes:", int(alice_cd4["significant"].sum()))

In [ ]:
# Founder subclone-family per donor + the tumor fraction / dominance it recovers.
# Malignant donors: founder = dominant clone. Non-malignant donors: founder = LARGEST clonotype.
# The malignant family = founder + its ALICE-SIGNIFICANT <=1-aa variants (non-significant neighbors
# are coincidental sequence similarity -> dropped). family_frac/fold ask whether collapsing that
# significant family crosses the nb18 threshold (dom_frac >= FRAC_THRESH AND fold >= RATIO_THRESH).
def _dominance(sub, members):
    """Collapse `members` into one clone; return (n, total, dom_frac, fold_change) vs the rest."""
    total = sub["n_cells"].sum()
    n = sub.loc[sub["cdr3"].isin(members), "n_cells"].sum()
    rest = sub.loc[~sub["cdr3"].isin(members), "n_cells"]
    second = int(rest.max()) if len(rest) else 0
    dom_frac = n / total if total else 0.0
    fold = (n / second) if second else np.inf
    return int(n), total, dom_frac, fold


rows = []
for donor, sub in clono_cd4.groupby("donor", observed=True):
    is_mal = bool(sub["is_founder"].any())
    if is_mal:
        founders = set(sub.loc[sub["is_founder"], "cdr3"]); src = "dominant"
    else:  # non-malignant: seed on the largest clonotype
        founders = {sub.sort_values("n_cells", ascending=False)["cdr3"].iloc[0]}; src = "largest_clonotype"
    sig = set(alice_cd4.loc[(alice_cd4["donor"] == donor) & alice_cd4["significant"], "cdr3"])
    fam = A.founder_family(sub, seeds=founders)
    fam_sig = set(founders) | (fam & sig)        # founder + significant <=1-aa variants

    ex_n, total, ex_frac, ex_fold = _dominance(sub, founders)
    fam_n, _, fam_frac, fam_fold = _dominance(sub, fam_sig)
    exact_call = (ex_frac >= FRAC_THRESH) and (ex_fold >= RATIO_THRESH)
    family_call = (fam_frac >= FRAC_THRESH) and (fam_fold >= RATIO_THRESH)
    rows.append({
        "donor": donor, "founder_source": src, "founder": "; ".join(sorted(founders)),
        "n_variant_clones": max(len(fam) - len(founders), 0),
        "n_family_significant": len(fam & sig),
        "exact_frac": ex_frac, "family_frac": fam_frac, "delta_frac": (fam_n - ex_n) / total,
        "exact_fold": ex_fold, "family_fold": fam_fold,
        "exact_call": exact_call, "family_call": family_call,
        "flipped_by_family": (not is_mal) and family_call})

use1 = (pd.DataFrame(rows)
        .merge(clone_summary[["donor", "study", "disease", "disease_stage", "entity", "malignant"]],
               on="donor", how="left")
        .sort_values(["flipped_by_family", "delta_frac"], ascending=False))
use1.to_csv(USE1_CSV, index=False)
print("wrote", USE1_CSV, "| donors:", len(use1),
      "| non-malignant flipped by family:", int(use1["flipped_by_family"].sum()))
use1

## Use 2 — cross-patient convergence (expected negative)

Pool one founder per malignant donor into a single repertoire and test for ≤1-aa neighborhood
enrichment **across patients** (shared antigen / superantigen). Per Gniadecki the expectation is a
clean negative; one clone per donor is sparse, so this is underpowered — a companion to GLIPH2, not a
substitute. A positive here would be the surprising, report-worthy result.

In [ ]:
# Cross-patient: pool founders, test neighborhood enrichment across patients.
founders = (clono_cd4[clono_cd4["is_founder"]]
            .drop_duplicates(["donor", "cdr3"])[["donor", "cdr3"]].copy())
pool = founders[["cdr3"]].copy(); pool["is_founder"] = False  # no within-pool founder; test degree only
use2 = A.alice_test(pool, pgen, Q=Q, alpha=ALPHA)
use2 = use2.merge(founders[["donor", "cdr3"]], on="cdr3", how="left")
use2.to_csv(USE2_CSV, index=False)
print("pooled founders:", len(founders),
      "| cross-patient significant:", int(use2["significant"].sum()))
use2.head(10)

## Use 3 — reactive antigen-driven clusters in the CD8 infiltrate

ALICE on the **CD8** T cells of the same donors — per donor and pooled per disease stage — to surface
convergent, antigen-driven reactive TIL clusters (anti-tumor / viral / *S. aureus*). Optional: annotate
significant hits against VDJdb / McPAS specificity (needs the DB file). **Heavy** — cached.

In [ ]:
# Reactive CD8 infiltrate: ALICE per donor and pooled per disease stage. Heavy -> cached.
# Stale-cache guard: recompute when the cached donor set != the current clono_cd8 donor set.
stage = obs[["donor", "disease_stage"]].astype(str).drop_duplicates().set_index("donor")["disease_stage"]
clono_cd8 = clono_cd8.assign(disease_stage=clono_cd8["donor"].map(stage).astype(str))
alice_cd8 = pd.read_parquet(ALICE_CD8_PARQUET) if ALICE_CD8_PARQUET.exists() else None
cached_donors = set(alice_cd8.loc[alice_cd8["scope"] == "donor", "donor"]) if alice_cd8 is not None else None
if alice_cd8 is None or cached_donors != set(clono_cd8["donor"]):
    if alice_cd8 is not None:
        print("CD8 cache stale (donor set changed) -> recompute")
    per_donor = A.run_alice_by_group(clono_cd8, pgen, group="donor", Q=Q, alpha=ALPHA)
    per_donor["scope"] = "donor"
    by_stage = A.run_alice_by_group(clono_cd8, pgen, group="disease_stage", Q=Q, alpha=ALPHA)
    by_stage["scope"] = "stage"
    alice_cd8 = pd.concat([per_donor, by_stage], ignore_index=True)
    alice_cd8.to_parquet(ALICE_CD8_PARQUET, index=False)
    print("wrote", ALICE_CD8_PARQUET)
else:
    print("loaded cached", ALICE_CD8_PARQUET)
use3 = alice_cd8[alice_cd8["significant"]].sort_values("obs_deg", ascending=False)
use3.to_csv(USE3_CSV, index=False)
print("CD8 donors:", alice_cd8.loc[alice_cd8['scope'] == 'donor', 'donor'].nunique(),
      "| reactive CD8 significant clonotypes:", len(use3))
use3

In [ ]:
# CD8 coverage: every donor (significant or not), with clonotype count + #significant hits.
cd8_cov = (alice_cd8[alice_cd8["scope"] == "donor"]
           .groupby("donor", observed=True)
           .agg(n_clonotypes=("cdr3", "size"),
                n_with_neighbor=("obs_deg", lambda s: int((s >= 1).sum())),
                n_significant=("significant", "sum"))
           .reset_index()
           .merge(clone_summary[["donor", "study", "disease", "disease_stage", "malignant"]],
                  on="donor", how="left")
           .sort_values("n_significant", ascending=False))
print("CD8 donors:", len(cd8_cov), "| with >=1 reactive hit:", int((cd8_cov["n_significant"] > 0).sum()))
cd8_cov

## Figures

In [ ]:
# Figures (one finding each; figsize 5x3).
plt.rcParams["figure.dpi"] = 120

# 1) Use 1 — tumor fraction recovered by the ALICE family vs exact-match founder.
d = use1.sort_values("family_frac", ascending=False)
fig, ax = plt.subplots(figsize=(5, 3))
ax.scatter(d["exact_frac"], d["family_frac"], s=18)
lim = [0, max(d["family_frac"].max(), d["exact_frac"].max()) * 1.05]
ax.plot(lim, lim, "k--", lw=0.8)
ax.set_xlabel("exact-match founder frac"); ax.set_ylabel("ALICE family frac")
ax.set_title("ALICE recovers β-variant tumor cells")
fig.tight_layout(); fig.savefig(FIG_DIR / "alice_use1_family_vs_exact.png", bbox_inches="tight")

# 2) Use 1 — CD4 observed vs expected degree, founders highlighted.
m = alice_cd4.merge(clono_cd4[["donor", "cdr3", "is_founder"]], on=["donor", "cdr3"], how="left")
fig, ax = plt.subplots(figsize=(5, 3))
ax.scatter(m["exp_deg"] + 1e-9, m["obs_deg"], s=6, alpha=.3, label="all")
f = m[m["is_founder"].fillna(False)]
ax.scatter(f["exp_deg"] + 1e-9, f["obs_deg"], s=20, color="tab:red", label="founder")
ax.set_xscale("log"); ax.set_xlabel("expected degree (null)"); ax.set_ylabel("observed degree")
ax.set_title("CD4 clonotype neighborhood enrichment"); ax.legend(frameon=False, fontsize=8)
fig.tight_layout(); fig.savefig(FIG_DIR / "alice_use1_obs_vs_exp.png", bbox_inches="tight")

# 3) Use 2 — pooled cross-patient degree histogram (expected negative).
fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(use2["obs_deg"], bins=range(0, int(use2["obs_deg"].max()) + 2))
ax.set_xlabel("cross-patient neighbor degree"); ax.set_ylabel("founders")
ax.set_title(f"Cross-patient convergence: {int(use2['significant'].sum())} hits")
fig.tight_layout(); fig.savefig(FIG_DIR / "alice_use2_crosspatient_hist.png", bbox_inches="tight")

# 4) Use 3 — reactive CD8 significant clonotypes by stage.
st = use3[use3.get("scope", "") == "stage"] if "scope" in use3 else use3
if len(st) and "disease_stage" in st.columns:
    c = st.groupby("disease_stage").size().sort_values()
    fig, ax = plt.subplots(figsize=(5, 3))
    c.plot.barh(ax=ax)
    ax.set_xlabel("significant CD8 clonotypes"); ax.set_title("Reactive CD8 clusters by stage")
    fig.tight_layout(); fig.savefig(FIG_DIR / "alice_use3_reactive_by_stage.png", bbox_inches="tight")
plt.show()

In [ ]:
# Cache final ALICE malignancy -> adata.obs["tcr_malignant_alice"].
# Per donor: dominant CD4 clone + its ALICE-significant <=1-aa neighbors = malignant.
# Non-dominant donors -> not assigned (stay False), as before.
# Exception: Li2024_atlas__PT35 -> its top-2 largest CD4 clones (+ their significant neighbors).
PT35_EXCEPTION = "Li2024_atlas__PT35"
ALICE_MAL_PARQUET = OUT_DIR / "alice_malignancy_v2.parquet"

mal_sets = {}   # donor -> set of malignant CDR3 (seed clone(s) + significant <=1-aa neighbors)
for donor, sub in clono_cd4.groupby("donor", observed=True):
    sig = set(alice_cd4.loc[(alice_cd4["donor"] == donor) & alice_cd4["significant"], "cdr3"])
    if donor == PT35_EXCEPTION:
        seeds = set(sub.sort_values("n_cells", ascending=False)["cdr3"].iloc[:2])
    elif sub["is_founder"].any():
        seeds = set(sub.loc[sub["is_founder"], "cdr3"])
    else:
        continue   # no dominant clone -> leave benign (no malignancy call)
    fam = A.founder_family(sub, seeds=seeds)
    mal_sets[donor] = seeds | (fam & sig)

# Map malignant CDR3 sets back to CD4 cells (TRB from the unified clone key, as in clonotype_table).
trb = (adata.obs["tcr_clone_id"].astype(str)
       .str.extract(r"^TRB:([A-Z]+)$", expand=False).fillna(""))
trb = trb.mask(trb == "", adata.obs["trb_cdr3"].astype(str))
donor = adata.obs["donor"].astype(str)
is_cd4 = adata.obs["cell_type_T2"].isin(CD4_T_TYPES).to_numpy()

mal = np.zeros(adata.n_obs, dtype=bool)
for d, mset in mal_sets.items():
    mal |= is_cd4 & (donor == d).to_numpy() & trb.isin(mset).to_numpy()
adata.obs["tcr_malignant_alice"] = mal

pd.DataFrame({"cell_id": adata.obs_names, "tcr_malignant_alice": mal}).to_parquet(ALICE_MAL_PARQUET, index=False)
print("malignant donors:", len(mal_sets), "| malignant cells:", int(mal.sum()),
      "| PT35 malignant cells:", int((is_cd4 & (donor == PT35_EXCEPTION).to_numpy() & mal).sum()))
print("wrote", ALICE_MAL_PARQUET)

In [ ]:
pd.crosstab(adata.obs["tcr_malignant_alice"], adata.obs["cell_type_T2"], margins=True)

In [ ]:
# Save the full annotated adata (with tcr_malignant_alice) for reuse elsewhere.
ADATA_MALIG_H5AD = OUT_DIR / "skin_T_tcr_malig_v2.h5ad"
adata.write_h5ad(ADATA_MALIG_H5AD)
print("wrote", ADATA_MALIG_H5AD, "|", adata.shape)